# Config System Example

This notebook demonstrates the fluent `Config` builder in
`robotdataset.configuration_system`, used for configuring WorldModel /
VLA training runs and for generating W&B sweep configs.


## 1. Building a config with the fluent API

In [1]:
from robotdataset.configuration_system import Config

cfg = Config()
cfg.dataset(name="oxe", batch_size=32, shuffle=True).training(
    learning_rate=1e-4, num_epochs=100
)

cfg


/home/user/robotdataset/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Config({'dataset': {'name': 'oxe', 'batch_size': 32, 'shuffle': True}, 'training': {'learning_rate': 0.0001, 'num_epochs': 100}})

Groups and fields are learned dynamically: calling `cfg.dataset(...)`
creates the `dataset` group (if it doesn't exist yet) and records each
field's value and inferred type.


In [2]:
print(cfg.groups())
print(cfg.fields("dataset"))
print(cfg.dataset.batch_size, cfg.schema("dataset", "batch_size").type)


['dataset', 'training']
['name', 'batch_size', 'shuffle']
32 int


## 2. Loading a config from YAML

In [3]:
from pathlib import Path

config_path = Path("../robotdataset/configuration_system/example_config.yaml")
print(config_path.read_text())


dataset:
  name: oxe
  batch_size: 32
  shuffle: true

training:
  learning_rate:
    type: float
    bounds:
      min: 1.0e-5
      max: 1.0e-2
    default: 1.0e-4
  optimizer:
    type: str
    values: ["adam", "sgd", "adamw"]
  num_epochs: 100



In [4]:
cfg = Config.from_file(config_path)
cfg.to_dict()


{'dataset': {'name': 'oxe', 'batch_size': 32, 'shuffle': True},
 'training': {'learning_rate': {'type': 'float',
   'bounds': {'min': 1e-05, 'max': 0.01},
   'default': 0.0001},
  'optimizer': {'type': 'str', 'values': ['adam', 'sgd', 'adamw']},
  'num_epochs': 100}}

Fields declared with a `type` + `bounds`/`values` (like
`training.learning_rate` and `training.optimizer` above) are learned as
hyperparameters: they carry bounds/candidate values in addition to a
default, which makes them eligible for sweep export below.


In [5]:
lr_spec = cfg.schema("training", "learning_rate")
opt_spec = cfg.schema("training", "optimizer")
print(lr_spec)
print(opt_spec)


FieldSpec(name='learning_rate', type='float', bounds={'min': 1e-05, 'max': 0.01}, values=None, default=0.0001)
FieldSpec(name='optimizer', type='str', bounds=None, values=['adam', 'sgd', 'adamw'], default=None)


## 3. Updating and saving a config

In [6]:
cfg.training(num_epochs=200)  # plain fields can be updated the same way
cfg.save("/tmp/updated_config.yaml")
print(Path("/tmp/updated_config.yaml").read_text())


dataset:
  name: oxe
  batch_size: 32
  shuffle: true
training:
  learning_rate:
    type: float
    bounds:
      min: 1.0e-05
      max: 0.01
    default: 0.0001
  optimizer:
    type: str
    values:
    - adam
    - sgd
    - adamw
  num_epochs: 200



## 4. Exporting a W&B sweep config

In [7]:
sweep_config = cfg.to_sweep(
    method="bayes",
    metric={"name": "val_loss", "goal": "minimize"},
)
sweep_config


{'method': 'bayes',
 'parameters': {'dataset.name': {'value': 'oxe'},
  'dataset.batch_size': {'value': 32},
  'dataset.shuffle': {'value': True},
  'training.learning_rate': {'min': 1e-05, 'max': 0.01},
  'training.optimizer': {'values': ['adam', 'sgd', 'adamw']},
  'training.num_epochs': {'value': 200}},
 'metric': {'name': 'val_loss', 'goal': 'minimize'}}

- `bounds` fields (e.g. `training.learning_rate`) become continuous
  `min`/`max` ranges.
- `values` fields (e.g. `training.optimizer`) become discrete/categorical
  choices.
- Plain fields are exported as a fixed `value`, so the full config is
  still captured in the sweep.


In [8]:
cfg.to_sweep_file("/tmp/sweep.yaml", method="bayes")
print(Path("/tmp/sweep.yaml").read_text())


method: bayes
parameters:
  dataset.name:
    value: oxe
  dataset.batch_size:
    value: 32
  dataset.shuffle:
    value: true
  training.learning_rate:
    min: 1.0e-05
    max: 0.01
  training.optimizer:
    values:
    - adam
    - sgd
    - adamw
  training.num_epochs:
    value: 200



The resulting `sweep.yaml` can be passed directly to `wandb sweep`:

```bash
wandb sweep /tmp/sweep.yaml
```
